In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install 'git+https://github.com/facebookresearch/detectron2.git' --quiet
!pip install easyocr fastapi uvicorn pyngrok python-multipart --quiet

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 28.4 MB/s eta 0:00:00


In [1]:
import os, cv2, torch, random, io, uvicorn
import numpy as np
import matplotlib.cm as cm
from PIL import Image
from torchvision.ops import nms
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

import easyocr
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import nest_asyncio

In [2]:
COCO_OUT   = "/content/drive/MyDrive/Newspaper_Project/coco_dataset"
TRAIN_JSON = f"{COCO_OUT}/annotations/train.json"
VAL_JSON   = f"{COCO_OUT}/annotations/val.json"
MODEL_PATH = "/content/drive/MyDrive/Newspaper_Project/model_output/model_final.pth"
BERT_DIR   = "/content/drive/MyDrive/Newspaper_Project/news_classifier/final_model"

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Detectron2
for name in ["newspaper_train", "newspaper_val"]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

register_coco_instances("newspaper_train", {}, TRAIN_JSON, image_root="")
register_coco_instances("newspaper_val",   {}, VAL_JSON,   image_root="")
MetadataCatalog.get("newspaper_train").thing_classes = ["article", "ads", "headline"]
MetadataCatalog.get("newspaper_val").thing_classes   = ["article", "ads", "headline"]

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.NUM_CLASSES       = 3
cfg.MODEL.WEIGHTS                     = MODEL_PATH
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST   = 0.3
cfg.TEST.DETECTIONS_PER_IMAGE         = 100
predictor = DefaultPredictor(cfg)

# EasyOCR
reader = easyocr.Reader(['en'], gpu=(DEVICE == "cuda"))

# DistilBERT
tokenizer  = DistilBertTokenizerFast.from_pretrained(BERT_DIR, local_files_only=True)
bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_DIR, local_files_only=True).to(DEVICE)
bert_model.eval()

LABEL_MAP  = {0: "article", 1: "ads", 2: "headline"}
CATEGORIES = [
    "Politics", "Business & Economy", "Sports", "Entertainment",
    "Science & Technology", "Health & Medicine", "Crime & Law",
    "Education", "Environment", "Lifestyle", "Travel",
    "Religion & Culture", "Parenting", "Opinion & Editorial", "International"
]

print(f"✓ All models loaded on {DEVICE}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✓ All models loaded on cuda


In [4]:
def clean_instances(instances, iou_threshold=0.3):
    if len(instances) == 0:
        return instances
    boxes, scores, classes = instances.pred_boxes.tensor, instances.scores, instances.pred_classes
    keep = []
    for cls_id in classes.unique():
        mask = (classes == cls_id)
        idx  = nms(boxes[mask], scores[mask], iou_threshold)
        keep.extend(mask.nonzero(as_tuple=True)[0][idx].tolist())
    return instances[sorted(set(keep))]

def ocr_crop(img_bgr, box, padding=5):
    h, w = img_bgr.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1-padding), max(0, y1-padding)
    x2, y2 = min(w, x2+padding), min(h, y2+padding)
    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return ""
    return " ".join(reader.readtext(crop, detail=0, paragraph=True)).strip()

def classify_text(text):
    if not text or len(text) < 15:
        return None, 0.0
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding="max_length", max_length=128).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(bert_model(**inputs).logits, dim=1).squeeze()
    conf, pred = probs.max(0)
    return CATEGORIES[pred.item()], round(conf.item(), 3)

def analyze_newspaper(img_bgr):
    instances = clean_instances(predictor(img_bgr)["instances"].to("cpu"))
    results   = []
    for box, cls_id, score in zip(instances.pred_boxes.tensor.numpy(),
                                   instances.pred_classes.numpy(),
                                   instances.scores.numpy()):
        rtype = LABEL_MAP[int(cls_id)]
        text  = ocr_crop(img_bgr, box)
        category, conf = classify_text(text) if rtype == "article" else (None, 0.0)
        results.append({
            "box":        [round(float(v), 1) for v in box],
            "type":       rtype,
            "score":      round(float(score), 3),
            "text":       text[:300],
            "category":   category,
            "confidence": conf
        })
    return results

In [5]:
CATEGORY_COLORS_BGR = {
    cat: tuple(int(c * 255) for c in reversed(cm.tab20.colors[i][:3]))
    for i, cat in enumerate(CATEGORIES)
}
REGION_COLORS_BGR = {
    "ads":      (0,   0,   255),
    "headline": (0,   200, 0  ),
}

def draw_boxes(img_bgr, results):
    out = img_bgr.copy()
    for r in results:
        x1, y1, x2, y2 = [int(v) for v in r["box"]]
        label = r["type"]
        if label == "article" and r["category"]:
            color   = CATEGORY_COLORS_BGR[r["category"]]
            display = f"{r['category']} {r['confidence']:.0%}"
        elif label == "article":
            color, display = (128, 128, 128), "article"
        else:
            color   = REGION_COLORS_BGR[label]
            display = label.capitalize()
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 4)
        (tw, th), _ = cv2.getTextSize(display, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
        cv2.rectangle(out, (x1, y1-th-12), (x1+tw+8, y1), color, -1)
        cv2.putText(out, display, (x1+4, y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    return out

app = FastAPI(title="Newspaper Layout Analyzer")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def root():
    return {"status": "running",
            "endpoints": ["/analyze", "/analyze/visualize", "/health"]}

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}

@app.post("/analyze")
async def analyze(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        return JSONResponse(status_code=400, content={"error": "Upload an image file"})
    contents = await file.read()
    img_bgr  = cv2.imdecode(np.frombuffer(contents, np.uint8), cv2.IMREAD_COLOR)
    if img_bgr is None:
        return JSONResponse(status_code=400, content={"error": "Could not decode image"})
    results = analyze_newspaper(img_bgr)
    summary = {
        "total_regions": len(results),
        "articles":      sum(1 for r in results if r["type"] == "article"),
        "headlines":     sum(1 for r in results if r["type"] == "headline"),
        "ads":           sum(1 for r in results if r["type"] == "ads"),
        "categories_found": list({r["category"] for r in results if r["category"]})
    }
    return JSONResponse(content={"summary": summary, "regions": results})

@app.post("/analyze/visualize")
async def analyze_visualize(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        return JSONResponse(status_code=400, content={"error": "Upload an image file"})
    contents = await file.read()
    img_bgr  = cv2.imdecode(np.frombuffer(contents, np.uint8), cv2.IMREAD_COLOR)
    if img_bgr is None:
        return JSONResponse(status_code=400, content={"error": "Could not decode image"})
    results   = analyze_newspaper(img_bgr)
    annotated = draw_boxes(img_bgr, results)
    _, buffer = cv2.imencode(".png", annotated)
    return StreamingResponse(io.BytesIO(buffer.tobytes()), media_type="image/png")

print("✓ App defined — 3 endpoints ready")

✓ App defined — 3 endpoints ready


In [6]:
import asyncio
nest_asyncio.apply()

ngrok.set_auth_token("3FiSOV6hPD7STFCr6HaUdDpWkvj_6iZjSeyrqiaBnb6jot2EC")

try:
    public_url = ngrok.connect(8000)
    print(f"✓ Public URL : {public_url}")
    print(f"  Docs       : {public_url}/docs")
    print(f"  Analyze    : POST {public_url}/analyze")
    print(f"  Visualize  : POST {public_url}/analyze/visualize")
except Exception as e:
    print(f"Tunnel error: {e}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
loop   = asyncio.get_event_loop()
loop.create_task(server.serve())

✓ Public URL : NgrokTunnel: "https://conceded-reformat-craftily.ngrok-free.dev" -> "http://localhost:8000"
  Docs       : NgrokTunnel: "https://conceded-reformat-craftily.ngrok-free.dev" -> "http://localhost:8000"/docs
  Analyze    : POST NgrokTunnel: "https://conceded-reformat-craftily.ngrok-free.dev" -> "http://localhost:8000"/analyze
  Visualize  : POST NgrokTunnel: "https://conceded-reformat-craftily.ngrok-free.dev" -> "http://localhost:8000"/analyze/visualize


<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77>>